# HAR with CSI data: Visualization


## Necessary Imports

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import math, os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier


import plotly.express as px
import plotly.io as pio

from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F


# Functions

In [46]:
import os, re, math
import numpy as np
import pandas as pd
from datetime import datetime

ts_epoch_re = re.compile(r'^\d+(\.\d+)?$')
brackets_re = re.compile(r'\[(.*?)\]')

def parse_timestamp(s: str) -> pd.Timestamp:
    s = s.strip().strip('"')
    if ts_epoch_re.match(s):
        # epoch in seconds (float)
        return pd.to_datetime(float(s), unit='s', utc=False)
    # datetime string
    # try with microseconds
    for fmt in ("%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S"):
        try:
            return pd.to_datetime(datetime.strptime(s, fmt))
        except ValueError:
            continue
    # last resort: pandas parser
    return pd.to_datetime(s, errors='coerce')

def extract_sig_mode(payload: str) -> int | None:
    # payload is a list where the sig_mode is the 6th element
    try:
        sig_index = 5
        token = payload[sig_index]
        return int(token)
    except Exception:
        return None

def parse_csi_payload(payload: str) -> list[int] | None:
    m = brackets_re.search(payload)
    if not m:
        return None
    ints = m.group(1).strip().split()
    try:
        return [int(x) for x in ints]
    except Exception:
        return None

def iq_to_amp_phase(ints: list[int]) -> tuple[np.ndarray, np.ndarray]:
    # ints are interleaved pairs per subcarrier
    # ESP32 convention: (imag, real) for each complex
    imag = np.array(ints[0::2], dtype=np.float32)
    real = np.array(ints[1::2], dtype=np.float32)
    amp = np.sqrt(real * real + imag * imag)
    phase = np.arctan2(imag, real)
    return amp, phase

def keep_indices_for_sig_mode(n_complex: int, sig_mode: int) -> np.ndarray:
    # n_complex expected 128 for ESP32 20 MHz
    if sig_mode == 1:
        # 802.11n (HT), keep 56: [4..31] + [33..60]
        keep = list(range(4, 32)) + list(range(33, 61))
    else:
        # default to 802.11a/g, keep 52: [6..31] + [33..58]
        keep = list(range(6, 32)) + list(range(33, 59))
    return np.array(keep, dtype=np.int32)

def load_csi_rows(directory: str, cls_substring: str, verbose=True):
    rows = []
    meta = []   # per-row metadata: timestamp, sig_mode, file
    for file in os.listdir(directory):
        if not (file.endswith('.csv') and cls_substring in file):
            continue
        fp = os.path.join(directory, file)
        if verbose:
            print(f"Reading: {fp}")
        with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                # skip header-like lines
                if line.startswith('type') or line.startswith('timestamp'):
                    continue
                # first token is timestamp (before first comma)
                first_comma = line.find(',')
                if first_comma <= 0:
                    continue
                ts_raw = line[:first_comma]
                payload = line[first_comma + 1:].strip()
                payload_elements = payload.split(',')
                ts = parse_timestamp(ts_raw)
                sm = extract_sig_mode(payload_elements)
                ints = parse_csi_payload(payload)
                if sm is None or ints is None or len(ints) % 2 != 0:
                    if verbose:
                        print(f"Skipping malformed row in {file}: sig_mode={sm}, ints_len={len(ints) if ints else None}")
                    continue
                amp, phase = iq_to_amp_phase(ints)
                n_complex = amp.shape[0]
                idx = keep_indices_for_sig_mode(n_complex, sm)
                amp_k = amp[idx]
                phase_k = phase[idx]
                rows.append((amp_k, phase_k))
                meta.append({'timestamp': ts, 'sig_mode': sm, 'file': file})
    if not rows:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    
    # Stack into DataFrames
    amps = pd.DataFrame([r[0] for r in rows])
    phases = pd.DataFrame([r[1] for r in rows])
    meta_df = pd.DataFrame(meta)

    return amps, phases, meta_df

# Models

In [66]:

# import libraries

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import math

from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt

class WiFiSensingNet(nn.Module):
    """
    Deep Neural Network for WiFi Sensing Classification
    Input: CSI data with shape (batch_size, 10800)
    Output: Binary classification (empty vs working)
    """
    def __init__(self, input_size=10800, hidden_sizes=[2048, 1024, 512, 256], num_classes=2, dropout_rate=0.1):
        super(WiFiSensingNet, self).__init__()
        
        # Create layers dynamically
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)


class TransformerClassifier(nn.Module):
    def __init__(self, n_features, n_classes, d_model=128, nhead=8, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        x = self.input_proj(x)
        x = self.transformer(x)        # (batch, seq_len, d_model)
        out = self.fc(x[:, -1, :])     # use last timestep
        return out



class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=128, n_layers=2, n_classes=5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features,
                            hidden_size=hidden_size,
                            num_layers=n_layers,
                            batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, (h, c) = self.lstm(x)
        # take last hidden state
        out = self.fc(h[-1])
        return out


class CSIDataset(Dataset):
    def __init__(self, csv_file, seq_len=300, step=5):
        df = pd.read_csv(csv_file)

        # Keep only CSI features (amps + phases)
        self.feature_cols = df.filter(regex="^(amp_|phase_)").columns
        self.seq_len = seq_len
        self.step = step

        # Group by file so we don’t mix across sessions
        self.groups = list(df.groupby("file"))

        # Precompute valid start indices: (group_idx, start_row)
        self.indices = []
        for gi, (_, g) in enumerate(self.groups):
            n = len(g) - seq_len
            # step through every 5th row
            for i in range(0, n, step):
                self.indices.append((gi, i))

        # Cache labels as categorical codes
        self.label_map = {v: i for i, v in enumerate(df["label"].astype("category").cat.categories)}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        gi, i = self.indices[idx]
        g = self.groups[gi][1]

        feats = g[self.feature_cols].iloc[i:i+self.seq_len].to_numpy(dtype=np.float32, copy=True)
        label_str = g["label"].iloc[i+self.seq_len-1]
        label = self.label_map[label_str]

        return torch.from_numpy(feats), torch.tensor(label, dtype=torch.long)





def deeplearning_training(model, train_loader, test_loader, num_epochs, patience,criterion, optimizer, scheduler = None, model_name = "task1") :
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)    
    print(f"Using device: {device}")
    
    
    patience_counter = 0
    best_accuracy = 0

    train_losses = []
    train_accuracies = []
    val_accuracies = []

    print("Starting training...")
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, target)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += target.size(0)
            correct_train += (predicted == target).sum().item()
        
        # Validation phase
        model.eval()
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                _, predicted = torch.max(outputs.data, 1)
                total_val += target.size(0)
                correct_val += (predicted == target).sum().item()
        
        # Calculate accuracies
        train_accuracy = 100 * correct_train / total_train
        val_accuracy = 100 * correct_val / total_val
        avg_loss = running_loss / len(train_loader)
        
        train_losses.append(avg_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        
        # Print progress
        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, '
                f'Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%')
        
        # Early stopping
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), model_name + '.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
        
        if scheduler : scheduler.step()

    print(f'Training completed. Best validation accuracy: {best_accuracy:.2f}%')
    return {"train_losses": train_losses, "train_accuracies": train_accuracies, "val_accuracies": val_accuracies}



## Prepare Directories

## Collecting and Saving Compact CSI Data

In [48]:

training_directory = '../data/rpi_data/train'
testing_directory = '../data/rpi_data/test'
output_data_directory = '../data/compact_data'
classes = ['empty', 'sleep', 'work']

N_of_samples = 2000



In [49]:
# Define the directory path where the CSV files are located
old = False 

if old : 

    dfs = [] 
    amp_dfs, phase_dfs = [], []

    for cls in classes:
        dfs.append(get_csi(training_directory, cls, verbose=True))
        amp_df, phase_df = csi_to_amplitude_phase(dfs[-1])
        amp_df, phase_df = filter_df(amp_df), filter_df(phase_df)
        amp_dfs.append(amp_df)
        phase_dfs.append(phase_df)

    Xs = []
    Ys = []

    for i, (amp_df, phase_df) in enumerate(zip(amp_dfs, phase_dfs)):
        X1 = select_data_portion(amp_df, N_of_samples)
        X2 = select_data_portion(phase_df, N_of_samples)
        X = pd.concat([X1, X2], axis=1)
        Xs.append(X)
        Y = np.zeros(len(X), dtype=int) + i  # Assign class labels 0, 1, 2 for walk, noact, jog
        Ys.append(Y)

    X_training = pd.concat(Xs, axis=0, ignore_index=True)
    Y_training = np.concatenate(Ys)

    print("The number of entries found are: ")
    for i, cls in enumerate(classes):
        print(f"{cls}: {len(amp_dfs[i])}", end=" ")
    print()
    print()

    print("Shape of X_training: ", X_training.shape)
    print("Shape of Y_training: ", Y_training.shape)




for dirname, directory in zip(['train', 'test'], [training_directory, testing_directory]):
    dfs = []  # use a list for accumulation

    for cls in classes:
        amps, phases, meta_df = load_csi_rows(directory, cls, verbose=False)

        tmp_df = pd.concat([
            meta_df.reset_index(drop=True),
            amps.add_prefix("amp_"),
            phases.add_prefix("phase_")
        ], axis=1)
        tmp_df["label"] = cls

        dfs.append(tmp_df)

    # combine all classes into one DataFrame
    df = pd.concat(dfs, axis=0).reset_index(drop=True)

    # save to CSV
    df.to_csv(f"{output_data_directory}/{dirname}.csv", index=False)



## Loading Compact data


In [69]:

train_ds = CSIDataset(f"{output_data_directory}/train.csv", seq_len = 300)
test_ds = CSIDataset(f"{output_data_directory}/test.csv", seq_len = 300)

train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = False)

for X, y in train_loader:
    print(X.shape, y.shape)  # expect (batch, 300, n_features), (batch,)
    break
# Inspect shapes
for X, y in test_loader:
    print(X.shape, y.shape)  # expect (batch, 300, n_features), (batch,)
    break


torch.Size([32, 300, 112]) torch.Size([32])
torch.Size([32, 300, 112]) torch.Size([32])


### Cleaning

## PCA and T-SNE Visualization

In [70]:
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA

# ---------------------------------------------------------
# Flatten sequences into feature vectors for PCA
# Each sample is (seq_len, n_features) → flatten to (seq_len * n_features,)
X_train = [x.numpy().reshape(-1) for x, y in train_ds]
Y_train = [y.item() for x, y in train_ds]

X_test = [x.numpy().reshape(-1) for x, y in test_ds]
Y_test = [y.item() for x, y in test_ds]

X_train = np.vstack(X_train)
X_test = np.vstack(X_test)
Y_train = np.array(Y_train)
Y_test = np.array(Y_test)

# ---------------------------------------------------------
# Run PCA
pca = PCA(n_components=2)
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

# ---------------------------------------------------------
# Map labels to class names
labels = {i: classes[i] for i in range(len(classes))}
labels_str = {str(k): v for k, v in labels.items()}

Y_train_str = [str(y) for y in Y_train]
Y_test_str = [str(y) for y in Y_test]

# Color map for consistency
colour_map = px.colors.qualitative.Plotly
color_discrete_map = {str(i): colour_map[i] for i in range(len(classes))}

title = f"PCA visualization of {classes} dataset"

# ---------------------------------------------------------
# Training scatter
fig0 = px.scatter(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    color=Y_train_str,
    color_discrete_map=color_discrete_map,
    labels=labels,
    title=f"{title} (Train)"
)

# Testing scatter
fig1 = px.scatter(
    x=X_test_pca[:, 0],
    y=X_test_pca[:, 1],
    color=Y_test_str,
    color_discrete_map=color_discrete_map,
    labels=labels,
    title=f"{title} (Test)"
)

# ---------------------------------------------------------
# Replace legend names with human-readable labels
for fig in [fig0, fig1]:
    for trace in fig.data:
        if trace.name in labels_str:
            trace.name = labels_str[trace.name]
            trace.legendgroup = labels_str[trace.name]
    fig.update_traces(marker=dict(size=5), selector=dict(mode='markers'))
    fig.update_layout(
        xaxis_title="First Principal Component",
        yaxis_title="Second Principal Component",
    )

fig0.show()
fig1.show()


KeyboardInterrupt: 

In [ ]:
import numpy as np
import plotly.express as px
from sklearn.decomposition import PCA

# ---------------------------------------------------------
# Flatten each sequence into 1D vector (seq_len * n_features)
X_train = [x.numpy().reshape(-1) for x, y in train_ds]
Y_train = [y.item() for x, y in train_ds]

X_test = [x.numpy().reshape(-1) for x, y in test_ds]
Y_test = [y.item() for x, y in test_ds]

X_train = np.vstack(X_train)
X_test = np.vstack(X_test)
Y_train = np.array(Y_train)
Y_test = np.array(Y_test)

# ---------------------------------------------------------
# Run PCA (3D)
pca_3d = PCA(n_components=3)
pca_3d.fit(X_train)

X_train_pca = pca_3d.transform(X_train)
X_test_pca = pca_3d.transform(X_test)

# ---------------------------------------------------------
# Map labels to class names
labels = {i: classes[i] for i in range(len(classes))}
labels_str = {str(k): v for k, v in labels.items()}

Y_train_str = [str(y) for y in Y_train]
Y_test_str = [str(y) for y in Y_test]

# Color map
colour_map = px.colors.qualitative.Plotly
color_discrete_map = {str(i): colour_map[i] for i in range(len(classes))}

# ---------------------------------------------------------
# Training scatter (3D)
fig_train = px.scatter_3d(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    z=X_train_pca[:, 2],
    color=Y_train_str,
    color_discrete_map=color_discrete_map,
    labels=labels,
    title=f"3D PCA visualization of {classes} dataset (Train)"
)

# Testing scatter (3D)
fig_test = px.scatter_3d(
    x=X_test_pca[:, 0],
    y=X_test_pca[:, 1],
    z=X_test_pca[:, 2],
    color=Y_test_str,
    color_discrete_map=color_discrete_map,
    labels=labels,
    title=f"3D PCA visualization of {classes} dataset (Test)"
)

# ---------------------------------------------------------
# Relabel legends with class names
for fig in [fig_train, fig_test]:
    for trace in fig.data:
        if trace.name in labels_str:
            trace.name = labels_str[trace.name]
            trace.legendgroup = labels_str[trace.name]

    fig.update_traces(marker=dict(size=4, opacity=0.7))
    fig.update_layout(
        scene=dict(
            xaxis_title="First Principal Component",
            yaxis_title="Second Principal Component",
            zaxis_title="Third Principal Component"
        ),
        legend_title_text="Classes"
    )

fig_train.show()
fig_test.show()


## Model training

### Deep Learning

In [31]:

# X_train_tensor = torch.FloatTensor(X_train)
# X_test_tensor = torch.FloatTensor(X_test)
# y_train_tensor = torch.LongTensor(y_train)
# y_test_tensor = torch.LongTensor(y_test)

# # Create datasets and dataloaders
# train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# batch_size = 64
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
# num_classes = len(torch.unique(y_train_tensor))
# print(f"Training data shape: {X_train_tensor.shape}")
# print(f"Test data shape: {X_testing_real.shape}")
# print(f"Number of classes: {num_classes}")


for data_dict in [same_dict, split_dict] : 
    
    label = data_dict["label"]
    train_loader, test_loader = data_dict["train_loader"], data_dict["test_loader"]
    input_size = train_loader.dataset[0][0].shape[-1]
    num_classes = len(torch.unique(torch.tensor([y for _, y in train_loader.dataset])))
    
    print(f"Input size: {input_size}")
    print(f"Number of classes: {num_classes}")
    model = WiFiSensingNet(input_size=input_size, hidden_sizes=[204, 102, 51, 25], num_classes=num_classes, dropout_rate=0.1)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.00005, weight_decay=1e-5)
    # scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    scheduler = None 

    # Training parameters
    num_epochs = 100
    patience = 25

    model_name = f"human_presence_{label}"
    training_results = deeplearning_training(model, train_loader, test_loader, num_epochs, patience, criterion, optimizer, scheduler, model_name)
    
    data_dict["results"].append(training_results)
    data_dict["models"].append(model)


Input size: 216000
Number of classes: 3
Using device: cpu
Starting training...
Epoch [5/100], Loss: 0.7670, Train Acc: 70.26%, Val Acc: 41.86%
Epoch [10/100], Loss: 0.6585, Train Acc: 84.84%, Val Acc: 65.12%
Epoch [15/100], Loss: 0.6108, Train Acc: 89.21%, Val Acc: 63.95%
Epoch [20/100], Loss: 0.5948, Train Acc: 89.80%, Val Acc: 63.95%
Epoch [25/100], Loss: 0.5418, Train Acc: 95.92%, Val Acc: 62.79%
Epoch [30/100], Loss: 0.5333, Train Acc: 96.50%, Val Acc: 60.47%
Early stopping at epoch 34
Training completed. Best validation accuracy: 65.12%
Input size: 216000
Number of classes: 3
Using device: cpu
Starting training...
Epoch [5/100], Loss: 0.7154, Train Acc: 82.75%, Val Acc: 42.66%
Epoch [10/100], Loss: 0.6123, Train Acc: 91.38%, Val Acc: 52.29%
Epoch [15/100], Loss: 0.5534, Train Acc: 97.20%, Val Acc: 48.62%
Epoch [20/100], Loss: 0.5183, Train Acc: 96.74%, Val Acc: 50.46%
Epoch [25/100], Loss: 0.4863, Train Acc: 98.37%, Val Acc: 51.38%
Epoch [30/100], Loss: 0.4568, Train Acc: 99.53%, 

In [ ]:

# Load best model
model.load_state_dict(torch.load('human_precense.pth'))
model.eval()

# Get predictions for test set
all_predictions = []
all_targets = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

# Print classification report
print("\nDeep Learning Model Performance:")
print("="*50)
report_ann = classification_report(all_targets, all_predictions, output_dict=True, target_names= classes)
print(report_ann)

cm_ann = confusion_matrix(all_targets, all_predictions)
print(cm_ann)

# Calculate final accuracy
final_accuracy = 100 * sum([1 for i, j in zip(all_targets, all_predictions) if i == j]) / len(all_targets)
print(f"\nFinal Test Accuracy: {final_accuracy:.2f}%")



Deep Learning Model Performance:
{'empty': {'precision': 0.6635514018691588, 'recall': 0.5035460992907801, 'f1-score': 0.5725806451612904, 'support': 141.0}, 'sleep': {'precision': 0.6236559139784946, 'recall': 0.6987951807228916, 'f1-score': 0.6590909090909091, 'support': 83.0}, 'work': {'precision': 0.7402597402597403, 'recall': 0.8769230769230769, 'f1-score': 0.8028169014084507, 'support': 130.0}, 'accuracy': 0.6864406779661016, 'macro avg': {'precision': 0.6758223520357979, 'recall': 0.6930881189789163, 'f1-score': 0.6781628185535501, 'support': 354.0}, 'weighted avg': {'precision': 0.6823671038348381, 'recall': 0.6864406779661016, 'f1-score': 0.677414162727079, 'support': 354.0}}
[[ 71  34  36]
 [ 21  58   4]
 [ 15   1 114]]

Final Test Accuracy: 68.64%


### SVC

In [34]:

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import pickle

pipe = Pipeline([('pca', PCA(n_components=10)),('svc', SVC())])
# pipe = Pipeline([('svc', SVC())])
# pipe = Pipeline([('pca', KernelPCA(n_components=10)),('svc', SVC())])


pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)
pickle.dump(pipe, open("human_precense_svc.pkl","wb"))

report_svc = classification_report(y_test, pipe.predict(X_test), target_names = classes, output_dict=True)
print(report_svc)

cm_svc = confusion_matrix(y_test, pipe.predict(X_test))
print(cm_svc)



{'empty': {'precision': 0.6229508196721312, 'recall': 0.5390070921985816, 'f1-score': 0.5779467680608364, 'support': 141.0}, 'sleep': {'precision': 0.6666666666666666, 'recall': 0.7951807228915663, 'f1-score': 0.7252747252747253, 'support': 83.0}, 'work': {'precision': 0.7593984962406015, 'recall': 0.7769230769230769, 'f1-score': 0.7680608365019012, 'support': 130.0}, 'accuracy': 0.6864406779661016, 'macro avg': {'precision': 0.6830053275264665, 'recall': 0.7037036306710749, 'f1-score': 0.6904274432791543, 'support': 354.0}, 'weighted avg': {'precision': 0.6833084842327176, 'recall': 0.6864406779661016, 'f1-score': 0.6823056645187212, 'support': 354.0}}
[[ 76  33  32]
 [ 17  66   0]
 [ 29   0 101]]


### KNN

In [35]:
# pipe2 = Pipeline([('scaler', scaler),('pca', PCA(n_components=10)),('knn', KNeighborsClassifier())])
from doctest import REPORT_CDIFF


pipe2 = Pipeline([('pca', PCA(n_components=10)),('knn', KNeighborsClassifier())])
pipe2.fit(X_train, y_train)
pipe2.score(X_test, y_test)
pickle.dump(pipe2, open("human_precense_knn.pkl","wb"))

report_knn = classification_report(y_test, pipe2.predict(X_test), output_dict=True, target_names=classes)
## classification report
print(report_knn)

cm_knn = confusion_matrix(y_test, pipe2.predict(X_test))
## confusion matrix
print(cm_knn)


{'empty': {'precision': 0.5785714285714286, 'recall': 0.574468085106383, 'f1-score': 0.5765124555160143, 'support': 141.0}, 'sleep': {'precision': 0.6363636363636364, 'recall': 0.6746987951807228, 'f1-score': 0.6549707602339181, 'support': 83.0}, 'work': {'precision': 0.746031746031746, 'recall': 0.7230769230769231, 'f1-score': 0.734375, 'support': 130.0}, 'accuracy': 0.652542372881356, 'macro avg': {'precision': 0.6536556036556037, 'recall': 0.657414601121343, 'f1-score': 0.6552860719166441, 'support': 354.0}, 'weighted avg': {'precision': 0.6536183057369499, 'recall': 0.652542372881356, 'f1-score': 0.6528801675908847, 'support': 354.0}}
[[81 31 29]
 [24 56  3]
 [35  1 94]]


### Visualize model

In [ ]:

import seaborn as sns





def plot_results(results):


    epochs = list(range(1, len(results["train_losses"]) + 1))

    # Plot training loss
    plt.figure()
    plt.plot(epochs, results["train_losses"], marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title("Training Loss over Epochs")
    plt.grid(True)
    plt.show()

    # Plot training accuracy
    plt.figure()
    plt.plot(epochs, results["train_accuracies"], marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Training Accuracy (%)")
    plt.title("Training Accuracy over Epochs")
    plt.grid(True)
    plt.show()

    # Plot validation accuracy
    plt.figure()
    plt.plot(epochs, results["val_accuracies"], marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy (%)")
    plt.title("Validation Accuracy over Epochs")
    plt.grid(True)
    plt.show()



def compare_classification_results(reports, conf_matrices, model_names=None, target_names=None):
    """
    Compare classification reports and confusion matrices.

    Args:
        reports (list of dict): Output of sklearn.metrics.classification_report(..., output_dict=True)
        conf_matrices (list of np.ndarray): Confusion matrices
        model_names (list of str): Names of models
        target_names (list of str): Names of classes
    """
    if model_names is None:
        model_names = [f"Model {i+1}" for i in range(len(reports))]
    if target_names is None:
        target_names = list(reports[0].keys())
        target_names = [t for t in target_names if t not in ['accuracy', 'macro avg', 'weighted avg']]

    # Collect metrics into dataframe
    rows = []
    for name, report in zip(model_names, reports):
        for label in target_names:
            row = {
                'Model': name,
                'Class': label,
                'Precision': report[label]['precision'],
                'Recall': report[label]['recall'],
                'F1-Score': report[label]['f1-score'],
                'Support': report[label]['support']
            }
            rows.append(row)
    
    df_metrics = pd.DataFrame(rows)

    # --- Plot metrics ---
    for metric in ['Precision', 'Recall', 'F1-Score']:
        plt.figure()
        sns.barplot(data=df_metrics, x='Class', y=metric, hue='Model')
        plt.title(f'{metric} Comparison')
        plt.ylim(0, 1)
        plt.grid(axis='y')
        plt.legend(title='Model')
        plt.show()

    # --- Plot confusion matrices ---
    for name, cm in zip(model_names, conf_matrices):
        plt.figure()
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
        plt.title(f'Confusion Matrix: {name}')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.show()

    # --- Show metrics table ---
    print("\nSummary Table:")
    display(df_metrics)


reports = [report_ann, report_knn, report_svc]
conf_matrices = [cm_ann, cm_knn, cm_svc]
model_names = ['ANN', 'KNN', 'SVC']
target_names = ['empty', 'working']
compare_classification_results(reports, conf_matrices, model_names, target_names)


KeyError: 'working'

------